In [ ]:
# | echo: false

from pathlib import Path
from loguru import logger


class ProjectPath:
    @staticmethod
    def find_project_root(marker="pyproject.toml"):
        path = Path.cwd().resolve()
        while path != path.parent:
            if (path / marker).exists():
                return path
            path = path.parent
        raise FileNotFoundError(f"Could not find project root with marker: {marker}")

    ROOT = find_project_root()
    logger.info(f"Project root directory: {ROOT.resolve()}")
    assert ROOT.exists()

PROJECT_DIR = ProjectPath().ROOT

## Purpose

This notebook serves as an exploratory tool for examining the log files produced during the horse behavioural experiments conducted in October and November 2023. It facilitates the experimentation with text parsing techniques on the files before they are imported into a [DuckDB](https://duckdb.org) database. 

The primary objectives are:

1. To reconcile which log files should be included in or excluded from the analysis.
2. To conduct experiments with regular expressions ([`regex`](https://docs.python.org/3/howto/regex.html)) aimed at extracting pertinent data and fields from the log files.

## Experiment details and naming conventions

There are two main types of Experiment:

- Reward Prediction (RPE)
- Cognitive Bias (CB)

The RPE have the following subtypes:

- `RPE-A` : acquisition of response
- `RPE-H` : habit formation
- `RPE-E` : extinction of response
- `RPE-ER` : extinction prior to reinstatement of response
- `RPE-R` : reinstatement of response

### RPE-type experiments

A new experiment type (`RPE-ER`) was created during the experiments which was not in the original specification.

#### Logfile Exclusion rules

Logfiles that are from test runs and also bad data need to be excluded from the analysis.

Rules are **case-insensitive**. Files which satisfy the following conditions are excluded:

- All files with `_TEST_` as the subject name
- All files with `_FRECKLE_` as the subject name
- Some files with `_BONNIE_` (14 legitimate files - *TODO: CH to confirm details*)
- All files with `test` in the `Comment` field
- Possibly some logs with very short run-times - *TODO: CH to confirm the files that have been identified*
- All files with `_OLIVE_` as the subject name - probably discard, treat as optional for now (*TODO: CH to confirm treatment*)

#### Problems with log file names during experiments

- Appears that for greater than 20 trials (subjects?) the value in the filename was reported as `NaN`. e.g. see `pumba` experiments.
- Other `NaN`s? There are 31 files with `NaN` (some are restarts) - *TODO: Confirm with CH why were there restarts? Why not "new" experiment?*

#### Time differences

For each trial we calculate the following time differences: 

1. Time delta: (`touch` datetime - `start tone` datetime) 
2. Time delta: (`Next start` datetime - `dispense of pellets` datetime)
3. Time delta: (`Dispense final pellets` datetime - `start tone` datetime)

We use item 3 as cross check on the consistency of previous time deltas.

These calculated quantities are the same for all `RPE`-type experiments.

In [ ]:
# | echo: false

import pandas as pd
from horse_logic.logfiles import Logs
from horse_logic.utils import export_data_to_csv, set_custom_logger_format
from IPython.display import Markdown, display
import textwrap

In [ ]:
# | echo: false

import itables.options as opt
from itables import init_notebook_mode

init_notebook_mode(
    all_interactive=True, connected=False
)  # Display Pandas dataframes in a more friendly paginated manner
opt.pageLength = 10  # Display 20 rows per page

In [ ]:
# | echo: false

set_custom_logger_format()

In [ ]:
# | echo: false

# Custom notebook functions


def display_sessions_by_subject(df_summary, subject_df):
    session_summary = []
    for n_subject, subject_name in enumerate(subject_df["subject_name"]):
        df_ = df_summary.drop(columns=["suffix", "subject_name"])[df_summary["subject_name"] == subject_name]
        n_session = len(df_)
        session_summary.append((n_subject + 1, subject_name.capitalize(), n_session))
        if n_session == 0:
            logger.info(f"Subject {subject_name}: No experiments conducted")
        else:
            df_.loc[:, "time_dff"] = df_["datetime"].diff()
            display(Markdown(f"### {n_subject+1}. {subject_name.capitalize()}: {n_session} session(s)"))
            display(df_)
    return pd.DataFrame(session_summary, columns=["Subject number", "Subject name", "Session count"])

In [ ]:
# | echo: false


def get_subject_info():
    HORSE_ORDER_XLSX = "Cohort data for MB.xlsx"
    HORSE_ORDER_PATH = PROJECT_DIR / "docs" / "from_CH"
    HORSE_ORDER_FILEPATH = HORSE_ORDER_PATH / HORSE_ORDER_XLSX

    if HORSE_ORDER_FILEPATH.exists():
        subject_df = pd.read_excel(HORSE_ORDER_FILEPATH)
        logger.info(f"Loaded horse order info from: {HORSE_ORDER_FILEPATH}")

        subject_df.rename(
            {"No": "subject_number", "Horse": "subject_name"}, axis=1, inplace=True
        )  # Rename columns
        subject_df["subject_name"] = subject_df[
            "subject_name"
        ].str.lower()  # Ensure lower case names for later subject lookup
        return subject_df
    else:
        logger.error(f"Horse order info not found: {HORSE_ORDER_FILEPATH}")
        return None

### Load Subject info

In [ ]:
# | echo: false

subject_df = get_subject_info()

subject_list = textwrap.fill(
    str(sorted(subject_df["subject_name"].to_list())), width=90, initial_indent="\t"
)
logger.info(f"Subject count: {len(subject_df['subject_name'])}")
logger.info(f"Sorted subject names:\n{subject_list}")

# ensure lower case subject names for later lookup

subject_df["subject_name"] = subject_df["subject_name"].str.lower()

## Directory information: log file data and outputs

In [ ]:
# | echo: false

LOGFILES_DIR = PROJECT_DIR / "data/results/zips/data_17Jan2020_email_hillydale_equine"

assert LOGFILES_DIR.exists()
logger.info(f"Logfiles dir: {LOGFILES_DIR.resolve()}")

In [ ]:
# | echo: false

DATA_DIR = PROJECT_DIR / "data"
EXPERIMENT_TYPE = "RPE"  # or CB

assert DATA_DIR.exists()

In [ ]:
# | echo: false

OUTPUT_DIR = DATA_DIR / f"results/{EXPERIMENT_TYPE}"

assert OUTPUT_DIR.exists()
logger.info(f"Outputs dir: {OUTPUT_DIR.resolve()}")

## Log file reconciliation

### Initial exclusions (rule-based)

In [ ]:
# | echo: false

# Define the default set of log files to exclude before fine-tuning after
logs = Logs(path=LOGFILES_DIR, patterns=["*_test*.log", "*freckle*.log"], include=False)

In [ ]:
# | echo: false

logger.info(logs)
logger.info(f"Included logfiles: {len(logs.included_files)}")
logger.info(f"Excluded logfiles: {len(logs.excluded_files)}")

In [ ]:
# | echo: false

df_excluded = pd.DataFrame(logs.excluded_files, columns=["Excluded"])

In [ ]:
# | echo: false

df_excluded.sort_values(by="Excluded").reset_index(drop=True)

### Included log files

Initial list of log files that are included based on rules specified above.

In [ ]:
# | echo: false

df_included = pd.DataFrame(sorted(logs.included_files), columns=["Included"])

In [ ]:
# | echo: false

export_data_to_csv(
    df_included,
    "List of files to be included in analysis",
    OUTPUT_DIR / "included_files.csv",
)

In [ ]:
# | echo: false

df_included

In [ ]:
# | echo: false

df_summary = logs.create_filename_components_dataframe(logs.included_files).sort_values(by=["subject_name", "session_number"]).reset_index(drop=True)

### Sessions summary by subject name

In [ ]:
# | echo: false

session_summary_df = display_sessions_by_subject(df_summary, subject_df)

### Session Summary - by Subject name and Session count

In [ ]:
# | echo: false

print(session_summary_df.sort_values(by="Subject name").reset_index(drop=True))

In [ ]:
# | echo: false

export_data_to_csv(
    session_summary_df,
    "Session overview (subject session counts)",
    OUTPUT_DIR / "session_overview.csv",
)

In [ ]:
# | echo: false

df_summary.drop(labels=["suffix"], axis=1, inplace=True)
df_summary = df_summary[["original_filename", "subject_name", "experiment_type", "session_number", "datetime"]]

In [ ]:
# | echo: false

# Calculate time differences between Experiments to look for anomalies
df_summary.loc[:, "time_diff"] = df_summary["datetime"].diff()

# Convert session number to integers - not sure how we got some floats?
df_summary["session_number"] = df_summary["session_number"].astype(pd.Int64Dtype())

In [ ]:
# | echo: false

df_summary

### Specific analysis for Bonnie log files

Used some Bonnie log files for testing - not all experiments are valid to include

In [ ]:
# | echo: false

# Get all Experiments where Bonnie was the subject
df_bonnie = df_summary[df_summary["subject_name"] == "bonnie"]

In [ ]:
# | echo: false

df_bonnie[
    df_bonnie["time_diff"] < pd.Timedelta("5m")
]  # Look at those with less than 5 minutes between experiments

In [ ]:
# | echo: false

bonnie_exclude_list = sorted(
    df_bonnie[df_bonnie["time_diff"] < pd.Timedelta("5m")]["original_filename"].tolist()
)

In [ ]:
# | echo: false

export_data_to_csv(
    pd.DataFrame(sorted(bonnie_exclude_list)),
    "Bonnie log file exclude list",
    OUTPUT_DIR / "bonnie_excluded_files.csv",
)

In [ ]:
# | echo: false

logs.handle_specific_files(bonnie_exclude_list, include=False)

In [ ]:
extra_bonnie_logs_include = [
    "Experiment_2023-10-11T16:56:09.875604_bonnie_58_RPE-A.log",
    "Experiment_2023-10-16T16:55:54.839279_bonnie_68_RPE-A.log",
    "Experiment_2023-10-19T14:37:56.973616_bonnie_72_RPE-H.log",
]

# Include these previously excluded files as per Signal chat 30 June 2024

In [ ]:
# | echo: false

logs.handle_specific_files(extra_bonnie_logs_include, include=True)

### Write out final lists of excluded and included log files

In [ ]:
# | echo: false

# Export list of all excluded files - by rules and specific exclusions

export_data_to_csv(
    pd.DataFrame(sorted(logs.excluded_files)),
    "List of log files excluded",
    OUTPUT_DIR / "all_excluded_files.csv",
)

In [ ]:
# | echo: false

export_data_to_csv(df_summary, "Experiment summary", OUTPUT_DIR / "Experiment_Summary_Oct2023_included.csv")

In [ ]:
# | echo: false

# Export list of all included files - by rules and specific exclusions (inc Bonnie)

# df = pd.DataFrame(sorted(logs.included_files), columns=["Filename"])

export_data_to_csv(
    pd.DataFrame(sorted(logs.included_files)),
    "File list of log files included",
    OUTPUT_DIR / "all_included_files.csv",
)

Key output file here is `all_included_files.csv` which defines all of the log files that will be included in the load of data to the DuckDB database for further analysis.

The next notebook to run is `notebooks/logfile-to-database-RPE.ipynb`.